# 中文新聞主題分類 — TF-IDF vs BERT 微調 (Colab)

自爬中央社 (CNA) 新聞，比較**傳統機器學習 (TF-IDF + 邏輯回歸)** 與 **預訓練語言模型 (BERT 微調)** 兩種方法。

> **執行前請先開 GPU**：上方選單 → 編輯 → 筆記本設定 → 硬體加速器選 **GPU**。
> 從上往下依序執行即可，全程使用 repo 內附的 973 篇資料集，幾分鐘內可跑完。

## 1. 安裝套件並 clone repo

In [ ]:
!pip -q install jieba transformers
!git clone https://github.com/Upisofsht/Nlp2026.git
%cd Nlp2026/final_project
# 安裝中文字型，讓混淆矩陣的中文標籤正常顯示
!apt-get -qq install -y fonts-noto-cjk
import matplotlib.font_manager as fm
for _p in fm.findSystemFonts(fontpaths='/usr/share/fonts/opentype/noto'):
    fm.fontManager.addfont(_p)

## 2. 資料來源：爬蟲（可選）

資料集已隨 repo 附上 (`data/news_dataset.csv`，共 973 篇)，**預設直接使用**，結果可重現。

下面這格示範爬蟲確實可跑：解除註解會用 CNA API 抓「科技」分類 3 篇（約幾秒），不影響後面流程。

In [ ]:
from src.crawler import CATEGORIES, list_article_urls, parse_article, fetch

print('CNA 分類代碼 → 標籤：')
for code_, label in CATEGORIES.items():
    print(f'  {code_}: {label}')

# --- 想實際驗證爬蟲，解除以下註解（約幾秒）---
# urls = list_article_urls('ait', n_articles=3)
# for u in urls:
#     art = parse_article(fetch(u))
#     print('•', art['title'])

## 3. 載入資料 + 中文前處理（清理 → jieba 斷詞 → 去停用詞）

In [ ]:
import sys, pandas as pd
sys.path.insert(0, '.')
from src.preprocess import tokenize, load_stopwords

df = pd.read_csv('data/news_dataset.csv')
df['text'] = df['title'].fillna('') + ' ' + df['content'].fillna('')

stop = load_stopwords('stopwords.txt')
# 給 TF-IDF 用：斷詞後用空白接成字串；BERT 直接吃原始 text，不需斷詞
df['tokens_str'] = df['text'].apply(lambda t: ' '.join(tokenize(t, stop)))

print('資料筆數：', len(df))
print(df['category'].value_counts())
df[['category', 'text', 'tokens_str']].head(3)

## 4. 標籤編碼 + 切分 train / test（80 / 20，兩種方法共用同一切分）

In [ ]:
from sklearn.model_selection import train_test_split

labels = sorted(df['category'].unique())
label2idx = {l: i for i, l in enumerate(labels)}
df['y'] = df['category'].map(label2idx)

tr, te = train_test_split(df, test_size=0.2, stratify=df['y'], random_state=42)
print('train', len(tr), '/ test', len(te), '/ 類別數', len(labels))

## 5. 方法一：TF-IDF + 邏輯回歸（傳統機器學習）

用關鍵詞的 TF-IDF 權重當特徵，丟進邏輯回歸分類。簡單、訓練快、可解釋。

In [ ]:
from src.features import build_tfidf
from src.models import train_baseline
from src.evaluate import evaluate_predictions, plot_confusion_matrix

X_tr, vec = build_tfidf(tr['tokens_str'])
X_te = vec.transform(te['tokens_str'])

clf = train_baseline(X_tr, tr['category'], kind='logreg')
tfidf_pred = clf.predict(X_te)

m_tfidf = evaluate_predictions(te['category'], tfidf_pred)
print('TF-IDF:', f"acc={m_tfidf['accuracy']:.4f}  macroF1={m_tfidf['macro_f1']:.4f}")
print(m_tfidf['report'])
plot_confusion_matrix(te['category'], tfidf_pred, labels, out_path='confusion_baseline.png')

## 6. 方法二：BERT 微調（預訓練語言模型）

用 `hfl/chinese-roberta-wwm-ext`，在 BERT 上接一層分類層，直接吃整篇新聞內文，不需斷詞。

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = 'hfl/chinese-roberta-wwm-ext'
MAX_LEN = 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.texts = list(texts)
        self.labels = list(labels)
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding='max_length',
                        max_length=MAX_LEN, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = NewsDataset(tr['text'], tr['y'])
test_ds = NewsDataset(te['text'], te['y'])
print('train_ds', len(train_ds), '/ test_ds', len(test_ds))

### 微調訓練（3 epochs，需要 GPU，約幾分鐘）

In [ ]:
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels))

args = TrainingArguments(
    output_dir='bert_out',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    logging_steps=20,
    save_strategy='no',
    report_to='none',
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds)
trainer.train()

## 7. BERT 評估 + 混淆矩陣

In [ ]:
pred_logits = trainer.predict(test_ds).predictions
pred_idx = pred_logits.argmax(axis=1)
bert_pred = [labels[i] for i in pred_idx]
bert_true = [labels[i] for i in te['y']]

m_bert = evaluate_predictions(bert_true, bert_pred)
print('BERT:', f"acc={m_bert['accuracy']:.4f}  macroF1={m_bert['macro_f1']:.4f}")
print(m_bert['report'])
plot_confusion_matrix(bert_true, bert_pred, labels, out_path='confusion_bert.png')

## 8. 結果比較

同一份 973 篇資料、同樣 80/20 切分，兩種方法的成績比較。

In [ ]:
summary = pd.DataFrame([
    {'方法': 'TF-IDF + 邏輯回歸', 'Accuracy': round(m_tfidf['accuracy'], 4),
     'Macro-F1': round(m_tfidf['macro_f1'], 4)},
    {'方法': 'BERT 微調', 'Accuracy': round(m_bert['accuracy'], 4),
     'Macro-F1': round(m_bert['macro_f1'], 4)},
])
print(summary.to_string(index=False))
print()
print('觀察：BERT 明顯優於 TF-IDF baseline；但兩者對「財經」皆難分類')
print('原因：財經用詞與科技／政治／兩岸高度重疊，屬類別語意重疊的典型混淆')
summary